In [ ]:
!git clone https://github.com/fishaudio/Bert-VITS2.git

Cloning into 'Bert-VITS2'...
remote: Enumerating objects: 3908, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3908 (delta 0), reused 0 (delta 0), pack-reused 3905 (from 2)
Receiving objects: 100% (3908/3908), 10.54 MiB | 10.46 MiB/s, done.
Resolving deltas: 100% (2417/2417), done.


In [ ]:
import json
import os

# ---------------- 配置参数 ----------------
config_dir = "/content/Bert-VITS2/configs"  # 你想保存的目录
os.makedirs(config_dir, exist_ok=True)

config_path = os.path.join(config_dir, "config.json")

# 训练时的关键参数（根据你的训练设置调整）
config = {
    "data": {
        "sampling_rate": 44100,       # 训练时采样率
        "filter_length": 1024,
        "win_length": 1024,
        "hop_length": 512,
        "n_mel_channels": 128         # Mel通道数
    },
    "model": {
        "upsample_rates": [8, 8, 4, 2],  # 总乘积要等于 hop_length
        "segment_size": 16384
    }
}

# ---------------- 写入文件 ----------------
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=4)

print(f"✅ config.json 已生成在: {config_path}")

✅ config.json 已生成在: /content/Bert-VITS2/configs/config.json


In [ ]:
from google.colab import drive
import shutil
import os

# 1️⃣ 挂载 Google Drive
drive.mount('/content/drive')

# 2️⃣ 定义源路径（你 Drive 里 G_900.pth 的位置）
src_model_path = '/content/drive/MyDrive/G_900.pth'  # 根据你实际路径修改

# 3️⃣ 定义目标路径（Bert-VITS2 里的 Model 文件夹）
dst_model_dir = '/content/Bert-VITS2/Cantonese_Model/Model'
os.makedirs(dst_model_dir, exist_ok=True)
dst_model_path = os.path.join(dst_model_dir, 'G_900.pth')

# 4️⃣ 拷贝文件
shutil.copy(src_model_path, dst_model_path)

print(f"✅ 模型已拷贝到 {dst_model_path}")

Mounted at /content/drive
✅ 模型已拷贝到 /content/Bert-VITS2/Cantonese_Model/Model/G_900.pth


In [ ]:
import torch
import json
import numpy as np
import sys
sys.path.append("/content/Bert-VITS2/for_deploy")  # 添加 infer.py 所在目录到搜索路径
from infer import infer, get_net_g

FileNotFoundError: [Errno 2] No such file or directory: 'config.yml'

In [ ]:


# ----------------------
# 配置文件加载
# ----------------------
config_path = "/content/Bert-VITS2/config.json"
with open(config_path, "r") as f:
    hps = json.load(f)

# 将json转对象风格访问
class HParams:
    def __init__(self, d):
        for k, v in d.items():
            if isinstance(v, dict):
                v = HParams(v)
            setattr(self, k, v)

hps = HParams(hps)
hps.version = "2.2"  # 确认你的版本号

# ----------------------
# 设备
# ----------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------
# 加载模型
# ----------------------
model_path = "/content/Bert-VITS2/Cantonese_Model/Model/G_900.pth"
net_g = get_net_g(model_path, hps.version, device, hps)

# ----------------------
# 文本准备
# ----------------------
text = "你好，這是一個測試"
sid = "0"  # 说话人id，对应 config.json 的 spk2id
language = "ZH"

# 如果没有BERT可以暂用随机张量
bert = {"ZH": type("BertDummy", (), {"get_bert_feature": lambda self, t, w2p, dev: torch.randn(1024, len(t))})()}

# ----------------------
# 推理生成音频
# ----------------------
audio = infer(
    text=text,
    emotion=None,
    sdp_ratio=0.8,
    noise_scale=0.667,
    noise_scale_w=0.8,
    length_scale=1.0,
    sid=sid,
    language=language,
    hps=hps,
    net_g=net_g,
    device=device,
    bert=bert,
)

# ----------------------
# 保存 WAV
# ----------------------
from scipy.io.wavfile import write
write("/content/test_output.wav", 22050, audio)